<a href="https://colab.research.google.com/github/cdiegor/OtimizacaoCombinatoria/blob/main/Programa%C3%A7%C3%A3o_Din%C3%A2mica_Exemplos_Pr%C3%A1ticos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Programação Dinâmica — Notebook Didático
**Data:** 2025-11-19

Notebook com aplicações clássicas de PD + visualizações com matplotlib (sem bibliotecas extras).


In [ ]:

import math
import random
from functools import lru_cache
from typing import List, Tuple
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


## 1) Fibonacci com memoização

**Enunciado formal — Fibonacci**  
**Entrada:** inteiro $n\ge 0$.  
**Saída:** o $n$-ésimo número de Fibonacci $F(n)$, definido por $F(0)=0, F(1)=1$ e $F(n)=F(n-1)+F(n-2)$ para $n\ge2$.  
**Objetivo:** computar $F(n)$ minimizando recomputações.  
**Modelo:** custo por operação unitário; complexidade em função de $n$.  
**Observação:** solução PD top-down (memoização) reduz custo de $\Theta(\varphi^n)$ para $\Theta(n)$.

<!-- descricao: fib -->

In [ ]:

from functools import lru_cache
@lru_cache(maxsize=None)
def fib(n: int) -> int:
    if n <= 1: return n
    return fib(n-1) + fib(n-2)
print([fib(i) for i in range(10)])


## 2) Troco mínimo — visualização de dp[v]

**Enunciado formal — Troco mínimo (Min Coins)**  
**Entrada:** conjunto de moedas $\mathcal{C}\subset \mathbb{N}_{>0}$ e valor alvo $V\in\mathbb{N}_{\ge0}$.  
**Saída:** número mínimo de moedas cujo somatório é $V$; opcionalmente, uma decomposição ótima.  
**Objetivo:** minimizar $|S|$ tal que $\sum_{c\in S} c = V$.  
**Restrições:** moedas ilimitadas, ordem irrelevante.  
**Modelo:** PD bottom-up com estado $dp[v]$ e transição $dp[v]=1+ min_{c\in\mathcal{C},\ v\ge c} \ dp[v-c]$.  
**Complexidade:** $O(V\cdot |\mathcal{C}|)$ tempo e $O(V)$ espaço.


In [ ]:

def min_coins(coins: List[int], V: int):
    INF = 10**9
    dp = [INF]*(V+1)
    prev = [-1]*(V+1)
    dp[0] = 0
    for v in range(1, V+1):
        for c in coins:
            if v-c >= 0 and dp[v] > dp[v-c] + 1:
                dp[v] = dp[v-c] + 1
                prev[v] = c
    sol = []
    x = V
    while x > 0 and prev[x] != -1:
        sol.append(prev[x])
        x -= prev[x]
    sol.reverse()
    return dp, sol

coins = [1,3,4]; V = 30
dp, sol = min_coins(coins, V)
print("Mínimo de moedas:", dp[V], "Uma solução:", sol)

plt.figure()
plt.plot(range(V+1), dp, marker="o", linewidth=1)
plt.title("Troco mínimo — dp[v]")
plt.xlabel("v")
plt.ylabel("dp[v]")
plt.show()


## 3) Distância de edição — matriz de DP

**Enunciado formal — Distância de Edição (Levenshtein)**  
**Entrada:** cadeias $A$ (tamanho $n$) e $B$ (tamanho $m$).  
**Saída:** custo mínimo para transformar $A$ em $B$ via inserção, deleção e substituição (cada uma de custo 1).  
**Objetivo:** minimizar número de operações.  
**Modelo:** $dp[i][j]$ = custo p/ transformar $A[1..i]$ em $B[1..j]$.  
**Transições:**  
$dp[i][j]=\min\{dp[i-1][j]+1,\ dp[i][j-1]+1,\ dp[i-1][j-1]+[A_i\neq B_j]\}$.  
**Bases:** $dp[i][0]=i$, $dp[0][j]=j$.  
**Complexidade:** $O(nm)$ tempo e $O(nm)$ espaço (otimizável).

<!-- descricao: editdistance -->

In [ ]:

def edit_distance(A: str, B: str):
    n, m = len(A), len(B)
    dp = np.zeros((n+1, m+1), dtype=int)
    for i in range(n+1): dp[i,0] = i
    for j in range(m+1): dp[0,j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if A[i-1]==B[j-1] else 1
            dp[i,j] = min(dp[i-1,j]+1, dp[i,j-1]+1, dp[i-1,j-1]+cost)
    return dp

A,B = "aberto", "fechado"
dp_ed = edit_distance(A,B)
print("Distância:", dp_ed[-1,-1])

plt.figure()
plt.imshow(dp_ed)
plt.title(f"Edit Distance — '{A}' vs '{B}'")
plt.xlabel("j"); plt.ylabel("i")
plt.colorbar()
plt.show()


## 4) Mochila 0/1 — curva valor × capacidade

**Enunciado formal — Mochila 0/1**  
**Entrada:** itens $i=1..n$ com pesos $w_i\in\mathbb{N}$, valores $v_i\in\mathbb{R}_{\ge0}$, e capacidade $W\in\mathbb{N}$.  
**Saída:** valor máximo total $\sum v_i x_i$ com $\sum w_i x_i \le W$ e $x_i\in\{0,1\}$.  
**Objetivo:** maximização sujeta a restrição de capacidade.  
**Modelo:** PD 1D: $dp[c]=\max\{dp[c],\ v_i+dp[c-w_i]\}$ para $c=W..w_i$.  
**Complexidade:** $O(nW)$ tempo e $O(W)$ espaço.

<!-- descricao: knapsack -->

In [ ]:

def knapsack_01(weights: List[int], values: List[int], W: int):
    dp = [0]*(W+1)
    for w,v in zip(weights, values):
        for c in range(W, w-1, -1):
            dp[c] = max(dp[c], v + dp[c-w])
    return dp

weights = [1,3,4,5]; values=[1,4,5,7]; W=20
dp_kn = knapsack_01(weights, values, W)
print("Valor ótimo em W:", dp_kn[W])

plt.figure()
plt.plot(range(W+1), dp_kn, marker="s", linewidth=1)
plt.title("Mochila 0/1 — valor por capacidade")
plt.xlabel("Capacidade"); plt.ylabel("Valor ótimo")
plt.show()


## 5) Caminho mínimo em DAG (ordem topológica)

**Enunciado formal — Caminho Mínimo em DAG**  
**Entrada:** DAG $G=(V,E)$ com pesos $w:E\to\mathbb{R}$ e origem $s\in V$.  
**Saída:** distâncias mínimas $dist[v]$ de $s$ a cada $v$.  
**Objetivo:** minimizar custo acumulado ao longo do caminho.  
**Modelo:** ordenar topologicamente e relaxar arestas nessa ordem:  
$dist[v]=\min\{dist[v], dist[u]+w(u,v)\}$.  
**Complexidade:** $O(n+m)$.

<!-- descricao: dagsp -->

In [ ]:

from collections import defaultdict, deque

def dag_shortest_path(n: int, edges: List[Tuple[int,int,int]], s: int):
    adj = defaultdict(list); indeg=[0]*n
    for u,v,w in edges:
        adj[u].append((v,w)); indeg[v]+=1
    q = deque([i for i in range(n) if indeg[i]==0])
    topo = []
    while q:
        u=q.popleft(); topo.append(u)
        for v,_ in adj[u]:
            indeg[v]-=1
            if indeg[v]==0: q.append(v)
    INF=10**12; dist=[INF]*n; dist[s]=0
    for u in topo:
        if dist[u]>=INF: continue
        for v,w in adj[u]:
            if dist[v] > dist[u]+w: dist[v]=dist[u]+w
    return topo, dist

edges=[(0,1,2),(0,2,4),(1,2,1),(1,3,7),(2,4,3),(4,3,2),(3,5,1),(4,5,5)]
topo, dist = dag_shortest_path(6, edges, 0)
print("Topológica:", topo); print("Dist:", dist)


## 6) LIS (PD O(n^2)) — visualização

**Enunciado formal — LIS (Maior Subsequência Crescente)**  
**Entrada:** sequência $A[1..n]\in\mathbb{R}^n$.  
**Saída:** comprimento (e opcionalmente, uma subsequência) da maior subsequência estritamente crescente.  
**Modelo PD:** $dp[i]=1+\max\{dp[j]: j<i,\ A_j<A_i\}$; base $dp[i]=1$.  
**Complexidade:** $O(n^2)$ tempo e $O(n)$ espaço (há versão $O(n\log n)$ por busca binária).

<!-- descricao: lis -->

In [ ]:

def lis_n2(a: List[int]):
    n=len(a); dp=[1]*n; parent=[-1]*n; best=0; end=0
    for i in range(n):
        for j in range(i):
            if a[j] < a[i] and dp[j]+1 > dp[i]:
                dp[i]=dp[j]+1; parent[i]=j
        if dp[i] > best: best=dp[i]; end=i
    seq=[]; k=end
    while k!=-1: seq.append(a[k]); k=parent[k]
    return best, list(reversed(seq))

random.seed(42)
a=[random.randint(1,20) for _ in range(16)]
length, seq = lis_n2(a)
print("Seq:", a); print("LIS len:", length, "LIS:", seq)

plt.figure()
plt.plot(range(len(a)), a, marker="o")
plt.title("Sequência e comprimento da LIS (ilustrativo)")
plt.xlabel("índice"); plt.ylabel("valor")
plt.show()


## 7) Matrix Chain Multiplication (parentetização ótima)

**Enunciado formal — Parentetização ótima de produto de matrizes**  
**Entrada:** dimensões $p_0,p_1,\dots,p_n$ para matrizes $A_i$ de tamanho $p_{i-1}\times p_i$.  
**Saída:** custo mínimo de multiplicar $A_1\cdot A_2\cdots A_n$ e uma parentetização ótima.  
**Modelo PD:** $dp[i][j]=\min\_{i\le k<j}\{dp[i][k]+dp[k+1][j]+p_{i-1}p_k p_j\}$, base $dp[i][i]=0$.  
**Complexidade:** $O(n^3)$ tempo e $O(n^2)$ espaço.

<!-- descricao: matrixchain -->

In [ ]:

import numpy as np

def matrix_chain_order(p: List[int]):
    n=len(p)-1
    dp=np.zeros((n,n), dtype=int)
    split=np.zeros((n,n), dtype=int)
    BIG=10**12
    for L in range(2, n+1):
        for i in range(0, n-L+1):
            j=i+L-1
            dp[i,j]=BIG; best=i
            for k in range(i, j):
                cost = dp[i,k]+dp[k+1,j]+p[i]*p[k+1]*p[j+1]
                if cost < dp[i,j]:
                    dp[i,j]=cost; best=k
            split[i,j]=best
    return dp, split

def parens(split, i, j):
    if i==j: return f"A{i+1}"
    k=split[i,j]
    return f"({parens(split, i, k)}·{parens(split, k+1, j)})"

p=[10,30,5,60, 8, 90, 32, 4, 9]
dp, split = matrix_chain_order(p)
print("Custo mínimo:", dp[0, len(p)-2])
print("Parentetização:", parens(split, 0, len(p)-2))


## 8) Weighted Interval Scheduling

**Enunciado formal — Agendamento de Intervalos com Pesos**  
**Entrada:** intervalos $(s_i,f_i,v_i)$, ordenados por fim, e função $p(i)=\max\{j<i: f_j\le s_i\}$.  
**Saída:** subconjunto não sobreposto de máximo valor total.  
**Modelo PD:** $dp[i]=\max\{dp[i-1],\ v_i + dp[p(i)]\}$ com base $dp[0]=0$.  
**Complexidade:** $O(n\log n)$ com busca binária para $p(i)$ (ou $O(n^2)$ se feito ingênuo).

<!-- descricao: wis -->

In [ ]:

def weighted_interval_scheduling(intervals: List[Tuple[int,int,int]]):
    intervals = sorted(intervals, key=lambda x: x[1])
    n=len(intervals)
    finishes=[f for _,f,_ in intervals]
    p=[0]*n
    for i in range(n):
        j=i-1
        while j>=0 and finishes[j] > intervals[i][0]:
            j-=1
        p[i]=j
    dp=[0]*(n+1)
    for i in range(1, n+1):
        dp[i]=max(dp[i-1], intervals[i-1][2]+dp[p[i-1]+1])
    res=[]; i=n
    while i>0:
        if dp[i]==dp[i-1]:
            i-=1
        else:
            res.append(intervals[i-1]); i=p[i-1]+1
    res.reverse()
    return dp[n], res

intervals = [(1,3,5),(2,5,6),(4,6,5),(6,7,4),(5,8,11),(7,9,2)]
best, chosen = weighted_interval_scheduling(intervals)
print("Valor ótimo:", best, "Escolhidos:", chosen)
